# 🎙️ LipSync — synchronisation labiale gratuite

**Fais parler une photo ou une vidéo avec ton propre audio, même pour 20 minutes.**

### Comment l'utiliser (3 clics)
1. Menu **Exécution → Modifier le type d'exécution** → choisis **GPU T4** → **Enregistrer**. *(à faire une seule fois)*
2. Clique sur le bouton **▶** de la cellule ci-dessous.
3. Autorise l'accès à Google Drive quand la fenêtre s'ouvre, puis attends 5 à 10 minutes la première fois.
   Un lien **https://….gradio.live** apparaît : **clique dessus**, ton outil s'ouvre.

### Bon à savoir
- Tes vidéos sont enregistrées dans **Google Drive → LipSync → projets**.
- Si Colab se coupe, relance la cellule et utilise l'onglet **« Reprendre un projet »** : les morceaux déjà faits ne sont pas refaits.
- Laisse cet onglet Colab ouvert pendant le traitement.


In [ ]:
#@title ▶️ Lancer LipSync
SAUVEGARDER_SUR_DRIVE = True  #@param {type:"boolean"}

import os, subprocess, sys, json, textwrap

def etape(msg):
    print("\n" + "=" * 60 + "\n" + msg + "\n" + "=" * 60, flush=True)

# 1. Carte graphique
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
if not gpu:
    raise SystemExit("❌ Pas de carte graphique. Menu Exécution → Modifier le type d'exécution → GPU T4 → "
                     "Enregistrer, puis relance cette cellule. Si la T4 n'est pas disponible, réessaie plus tard.")
gpu_name, gpu_mem = [x.strip() for x in gpu.splitlines()[0].split(",")]
gpu_mem = int(gpu_mem)
etape(f"✅ Carte graphique : {gpu_name} ({gpu_mem // 1024} Go)")

# 2. Google Drive
BASE = "/content/LipSync"
if SAUVEGARDER_SUR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/LipSync"
os.makedirs(BASE, exist_ok=True)

# 3. Installation (une seule fois par session)
REPO = "/content/LatentSync"
MARK = "/content/.lipsync_ok"
if not os.path.exists(MARK):
    etape("⏳ Installation (5 à 10 minutes la première fois)…")
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", "https://github.com/bytedance/LatentSync.git", REPO], check=True)
        subprocess.run(["git", "-C", REPO, "checkout", "-q", "a229c3948406bc2cf6eaf4873e662e70c6a04746"], check=True)
    pkgs = ["diffusers==0.32.2", "transformers==4.48.0", "decord==0.6.0", "einops", "omegaconf==2.3.0",
            "kornia==0.8.0", "insightface==0.7.3", "onnxruntime-gpu", "DeepCache==0.1.1", "ffmpeg-python",
            "imageio", "imageio-ffmpeg", "soundfile", "accelerate", "gradio"]
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise SystemExit("❌ L'installation a échoué. Fais une capture d'écran de ce message et envoie-la à Claude.")
    test = subprocess.run([sys.executable, "-c",
        "import decord, insightface, onnxruntime, diffusers, DeepCache, kornia, soundfile, ffmpeg, gradio"],
        capture_output=True, text=True, cwd=REPO)
    if test.returncode != 0:
        print(test.stderr[-3000:])
        raise SystemExit("❌ Un module ne se charge pas. Fais une capture d'écran et envoie-la à Claude.")
    open(MARK, "w").write("ok")
else:
    etape("✅ Déjà installé")

# 4. Modèle : 1.6 (plus net) si la carte a assez de mémoire, sinon 1.5 (T4 gratuite)
if gpu_mem >= 20000:
    HF_REPO, CONFIG, VERSION = "ByteDance/LatentSync-1.6", "configs/unet/stage2_512.yaml", "1.6"
else:
    HF_REPO, CONFIG, VERSION = "ByteDance/LatentSync-1.5", "configs/unet/stage2.yaml", "1.5"
ck = os.path.join(REPO, "checkpoints")
vfile = os.path.join(ck, "version.txt")
if not (os.path.exists(os.path.join(ck, "latentsync_unet.pt")) and os.path.exists(vfile)
        and open(vfile).read() == VERSION):
    etape(f"⏳ Téléchargement du modèle LatentSync {VERSION}…")
    from huggingface_hub import hf_hub_download
    for f in ("latentsync_unet.pt", "whisper/tiny.pt"):
        hf_hub_download(HF_REPO, f, local_dir=ck)
    open(vfile, "w").write(VERSION)
etape(f"✅ Modèle LatentSync {VERSION} prêt")

# 5. Fichiers de l'outil
APP = "/content/lipsync_outil"
os.makedirs(APP, exist_ok=True)
SOURCES = {"lipsync_core.py": "\"\"\"Préparation, découpage et assemblage des vidéos pour la synchro labiale.\n\nAucune dépendance au GPU ici : uniquement ffmpeg. Le modèle tourne dans\nlipsync_runner.py.\n\"\"\"\nimport json\nimport math\nimport os\nimport shutil\nimport subprocess\n\nFPS = 25\nIMAGE_EXT = {\".jpg\", \".jpeg\", \".png\", \".webp\", \".bmp\"}\nBOOMERANG_MAX_SEC = 30  # au-delà, simple boucle (l'inversion prendrait trop de mémoire)\n\n\ndef run(cmd):\n    res = subprocess.run(cmd, capture_output=True, text=True)\n    if res.returncode != 0:\n        raise RuntimeError(\"ffmpeg a échoué :\\n\" + \" \".join(cmd) + \"\\n\" + res.stderr[-1500:])\n    return res\n\n\ndef duration(path):\n    out = run([\"ffprobe\", \"-v\", \"error\", \"-show_entries\", \"format=duration\",\n               \"-of\", \"default=nw=1:nk=1\", path]).stdout.strip()\n    return float(out)\n\n\ndef frame_count(path):\n    out = run([\"ffprobe\", \"-v\", \"error\", \"-count_frames\", \"-select_streams\", \"v:0\",\n               \"-show_entries\", \"stream=nb_read_frames\", \"-of\", \"default=nw=1:nk=1\", path]).stdout.strip()\n    return int(out)\n\n\ndef is_image(path):\n    return os.path.splitext(path)[1].lower() in IMAGE_EXT\n\n\ndef _scale(max_h):\n    return f\"scale=-2:'min({max_h},trunc(ih/2)*2)'\"\n\n\nENC = [\"-c:v\", \"libx264\", \"-preset\", \"veryfast\", \"-crf\", \"18\", \"-pix_fmt\", \"yuv420p\"]\n\n\ndef make_base(face_path, out_path, max_h, audio_sec):\n    \"\"\"Vidéo de base à 25 i/s, redimensionnée. Boomerang (aller-retour) si courte.\"\"\"\n    tmp = out_path + \".fwd.mp4\"\n    if is_image(face_path):\n        run([\"ffmpeg\", \"-y\", \"-loop\", \"1\", \"-i\", face_path, \"-t\", \"4\", \"-r\", str(FPS),\n             \"-vf\", _scale(max_h), *ENC, \"-an\", out_path])\n        return out_path\n    run([\"ffmpeg\", \"-y\", \"-i\", face_path, \"-vf\", f\"fps={FPS},\" + _scale(max_h), *ENC, \"-an\", tmp])\n    src_sec = duration(tmp)\n    if src_sec >= audio_sec or src_sec > BOOMERANG_MAX_SEC:\n        shutil.move(tmp, out_path)\n        return out_path\n    # aller + retour pour que la boucle ne « saute » pas\n    run([\"ffmpeg\", \"-y\", \"-i\", tmp, \"-filter_complex\",\n         \"[0:v]split[a][b];[b]reverse,trim=start_frame=1,setpts=PTS-STARTPTS[r];[a][r]concat=n=2:v=1[v]\",\n         \"-map\", \"[v]\", *ENC, out_path])\n    os.remove(tmp)\n    return out_path\n\n\ndef plan_chunks(total_frames, chunk_sec):\n    size = max(FPS * 5, int(round(chunk_sec * FPS)))\n    chunks, start, i = [], 0, 0\n    while start < total_frames:\n        n = min(size, total_frames - start)\n        chunks.append({\"idx\": i, \"start\": start, \"frames\": n})\n        start += n\n        i += 1\n    return chunks\n\n\ndef prepare_project(proj_dir, work_dir, face_path, audio_path, chunk_sec, max_h, log=print):\n    \"\"\"Crée la frise vidéo complète et les morceaux à traiter. Renvoie le manifeste.\"\"\"\n    os.makedirs(work_dir, exist_ok=True)\n    done_dir = os.path.join(proj_dir, \"morceaux_finis\")\n    os.makedirs(done_dir, exist_ok=True)\n\n    audio_sec = duration(audio_path)\n    total_frames = int(math.ceil(audio_sec * FPS))\n    chunks = plan_chunks(total_frames, chunk_sec)\n\n    manifest_path = os.path.join(proj_dir, \"manifest.json\")\n    if os.path.exists(manifest_path):\n        old = json.load(open(manifest_path))\n        if old.get(\"chunks\") == chunks and old.get(\"max_h\") == max_h:\n            log(\"Projet existant retrouvé : les morceaux déjà terminés seront gardés.\")\n        else:\n            log(\"Réglages différents de la dernière fois : on recommence les morceaux.\")\n            shutil.rmtree(done_dir)\n            os.makedirs(done_dir)\n\n    todo = [c for c in chunks if not os.path.exists(os.path.join(done_dir, f\"morceau_{c['idx']:04d}.mp4\"))]\n    log(f\"Audio : {audio_sec/60:.1f} min · {len(chunks)} morceaux · {len(chunks)-len(todo)} déjà faits.\")\n\n    jobs = []\n    if todo:\n        log(\"Préparation de la vidéo de base…\")\n        base = make_base(face_path, os.path.join(work_dir, \"base.mp4\"), max_h, audio_sec)\n        timeline = os.path.join(work_dir, \"frise.mp4\")\n        if not (os.path.exists(timeline) and frame_count(timeline) == total_frames):\n            log(\"Création de la frise vidéo à la longueur de l'audio…\")\n            run([\"ffmpeg\", \"-y\", \"-stream_loop\", \"-1\", \"-i\", base, \"-frames:v\", str(total_frames),\n                 \"-r\", str(FPS), *ENC, \"-an\", timeline])\n        seg_dir = os.path.join(work_dir, \"segments\")\n        os.makedirs(seg_dir, exist_ok=True)\n        for c in todo:\n            t0, dur = c[\"start\"] / FPS, c[\"frames\"] / FPS\n            v = os.path.join(seg_dir, f\"v_{c['idx']:04d}.mp4\")\n            a = os.path.join(seg_dir, f\"a_{c['idx']:04d}.wav\")\n            run([\"ffmpeg\", \"-y\", \"-ss\", f\"{t0:.4f}\", \"-i\", timeline, \"-frames:v\", str(c[\"frames\"]),\n                 \"-r\", str(FPS), *ENC, \"-an\", v])\n            run([\"ffmpeg\", \"-y\", \"-ss\", f\"{t0:.4f}\", \"-i\", audio_path, \"-af\", \"apad\",\n                 \"-t\", f\"{dur:.4f}\", \"-ar\", \"16000\", \"-ac\", \"1\", a])\n            jobs.append({\"idx\": c[\"idx\"], \"frames\": c[\"frames\"], \"video\": v, \"audio\": a,\n                         \"out\": os.path.join(done_dir, f\"morceau_{c['idx']:04d}.mp4\")})\n\n    manifest = {\"chunks\": chunks, \"max_h\": max_h, \"chunk_sec\": chunk_sec,\n                \"audio\": audio_path, \"face\": face_path, \"total_frames\": total_frames}\n    json.dump(manifest, open(manifest_path, \"w\"), indent=1)\n    json.dump(jobs, open(os.path.join(work_dir, \"jobs.json\"), \"w\"), indent=1)\n    return manifest, jobs\n\n\ndef assemble(proj_dir, work_dir, manifest, out_path, log=print):\n    done_dir = os.path.join(proj_dir, \"morceaux_finis\")\n    parts = [os.path.join(done_dir, f\"morceau_{c['idx']:04d}.mp4\") for c in manifest[\"chunks\"]]\n    missing = [p for p in parts if not os.path.exists(p)]\n    if missing:\n        raise RuntimeError(f\"{len(missing)} morceaux manquent encore.\")\n    lst = os.path.join(work_dir, \"liste.txt\")\n    with open(lst, \"w\") as f:\n        for p in parts:\n            f.write(\"file '\" + p.replace(\"'\", \"'\\\\''\") + \"'\\n\")\n    video_only = os.path.join(work_dir, \"video_seule.mp4\")\n    log(\"Assemblage des morceaux…\")\n    run([\"ffmpeg\", \"-y\", \"-f\", \"concat\", \"-safe\", \"0\", \"-i\", lst, \"-c\", \"copy\", video_only])\n    log(\"Ajout de l'audio original en pleine qualité…\")\n    tmp = out_path + \".tmp.mp4\"\n    run([\"ffmpeg\", \"-y\", \"-i\", video_only, \"-i\", manifest[\"audio\"], \"-map\", \"0:v\", \"-map\", \"1:a\",\n         \"-c:v\", \"copy\", \"-c:a\", \"aac\", \"-b:a\", \"192k\", \"-shortest\", \"-movflags\", \"+faststart\", tmp])\n    shutil.move(tmp, out_path)\n    return out_path\n", "lipsync_runner.py": "\"\"\"Charge LatentSync une seule fois puis traite tous les morceaux de jobs.json.\n\nUsage : python lipsync_runner.py JOBS.json --steps 20 --guidance 1.5 --config ... --ckpt ...\nMettre LIPSYNC_FAKE=1 pour tester sans GPU (copie la vidéo au lieu d'utiliser le modèle).\n\"\"\"\nimport argparse\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport traceback\n\nFPS = 25\n\n\ndef say(msg):\n    print(msg, flush=True)\n\n\ndef finalize(raw, out, frames):\n    \"\"\"Force exactement `frames` images pour que l'assemblage reste synchro.\"\"\"\n    tmp_local = raw + \".exact.mp4\"\n    cmd = [\"ffmpeg\", \"-y\", \"-loglevel\", \"error\", \"-i\", raw,\n           \"-vf\", f\"fps={FPS},tpad=stop_mode=clone:stop=100\", \"-frames:v\", str(frames),\n           \"-c:v\", \"libx264\", \"-preset\", \"veryfast\", \"-crf\", \"18\", \"-pix_fmt\", \"yuv420p\", \"-an\", tmp_local]\n    subprocess.run(cmd, check=True)\n    tmp_out = out + \".part.mp4\"\n    shutil.copy(tmp_local, tmp_out)\n    os.replace(tmp_out, out)  # le morceau n'est « fini » qu'une fois entièrement copié\n    os.remove(tmp_local)\n\n\ndef load_pipeline(config_path, ckpt_path, deepcache):\n    import torch\n    from omegaconf import OmegaConf\n    from diffusers import AutoencoderKL, DDIMScheduler\n    from latentsync.models.unet import UNet3DConditionModel\n    from latentsync.pipelines.lipsync_pipeline import LipsyncPipeline\n    from latentsync.whisper.audio2feature import Audio2Feature\n\n    config = OmegaConf.load(config_path)\n    # La T4 gratuite (capacité 7.5) gère le float16 : moins de mémoire, plus rapide.\n    dtype = torch.float16 if torch.cuda.get_device_capability()[0] >= 7 else torch.float32\n    scheduler = DDIMScheduler.from_pretrained(\"configs\")\n    whisper = \"checkpoints/whisper/small.pt\" if config.model.cross_attention_dim == 768 else \"checkpoints/whisper/tiny.pt\"\n    audio_encoder = Audio2Feature(model_path=whisper, device=\"cuda\", num_frames=config.data.num_frames,\n                                  audio_feat_length=config.data.audio_feat_length)\n    vae = AutoencoderKL.from_pretrained(\"stabilityai/sd-vae-ft-mse\", torch_dtype=dtype)\n    vae.config.scaling_factor = 0.18215\n    vae.config.shift_factor = 0\n    unet, _ = UNet3DConditionModel.from_pretrained(OmegaConf.to_container(config.model), ckpt_path, device=\"cpu\")\n    unet = unet.to(dtype=dtype)\n    pipe = LipsyncPipeline(vae=vae, audio_encoder=audio_encoder, unet=unet, scheduler=scheduler).to(\"cuda\")\n    if deepcache:\n        from DeepCache import DeepCacheSDHelper\n        helper = DeepCacheSDHelper(pipe=pipe)\n        helper.set_params(cache_interval=3, cache_branch_id=0)\n        helper.enable()\n    return pipe, config, dtype\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"jobs\")\n    ap.add_argument(\"--steps\", type=int, default=20)\n    ap.add_argument(\"--guidance\", type=float, default=1.5)\n    ap.add_argument(\"--config\", default=\"configs/unet/stage2.yaml\")\n    ap.add_argument(\"--ckpt\", default=\"checkpoints/latentsync_unet.pt\")\n    ap.add_argument(\"--seed\", type=int, default=1247)\n    ap.add_argument(\"--no-deepcache\", action=\"store_true\")\n    args = ap.parse_args()\n\n    jobs = json.load(open(args.jobs))\n    todo = [j for j in jobs if not os.path.exists(j[\"out\"])]\n    total = len(todo)\n    if not total:\n        say(\"ALLDONE\")\n        return\n    fake = os.environ.get(\"LIPSYNC_FAKE\") == \"1\"\n\n    say(\"STATUS Chargement du modèle (1 à 2 minutes)…\")\n    if not fake:\n        pipe, config, dtype = load_pipeline(args.config, args.ckpt, not args.no_deepcache)\n        from accelerate.utils import set_seed\n\n    for k, job in enumerate(todo, 1):\n        say(f\"START {k} {total} {job['idx']}\")\n        t0 = time.time()\n        raw = job[\"video\"] + \".raw.mp4\"\n        try:\n            if fake:\n                # imite le modèle : sortie avec 2 images de trop\n                subprocess.run([\"ffmpeg\", \"-y\", \"-loglevel\", \"error\", \"-i\", job[\"video\"], \"-i\", job[\"audio\"],\n                                \"-vf\", \"tpad=stop_mode=clone:stop=2\", \"-c:v\", \"libx264\", \"-preset\", \"veryfast\",\n                                \"-c:a\", \"aac\", raw], check=True)\n            else:\n                if args.seed != -1:\n                    set_seed(args.seed)\n                pipe(video_path=job[\"video\"], audio_path=job[\"audio\"], video_out_path=raw,\n                     num_frames=config.data.num_frames, num_inference_steps=args.steps,\n                     guidance_scale=args.guidance, weight_dtype=dtype,\n                     width=config.data.resolution, height=config.data.resolution,\n                     mask_image_path=config.data.mask_image_path, temp_dir=\"temp\")\n            finalize(raw, job[\"out\"], job[\"frames\"])\n            os.remove(raw)\n            for p in (job[\"video\"], job[\"audio\"]):\n                if os.path.exists(p):\n                    os.remove(p)\n        except Exception as e:\n            msg = str(e)\n            say(\"TRACE \" + traceback.format_exc().replace(\"\\n\", \" | \"))\n            if \"face\" in msg.lower():\n                say(f\"ERROR FACE {job['idx']}\")\n            elif \"out of memory\" in msg.lower():\n                say(\"ERROR OOM\")\n            else:\n                say(\"ERROR OTHER \" + msg[:300].replace(\"\\n\", \" \"))\n            sys.exit(1)\n        say(f\"DONE {k} {total} {time.time() - t0:.1f}\")\n    say(\"ALLDONE\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "lipsync_app.py": "\"\"\"Interface web (Gradio) de l'outil de synchronisation labiale LipSync.\"\"\"\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\n\nimport gradio as gr\n\nimport lipsync_core as core\n\nREPO = os.environ.get(\"LIPSYNC_REPO\", \"/content/LatentSync\")\nBASE = os.environ.get(\"LIPSYNC_BASE\", \"/content/LipSync\")\nWORK = os.environ.get(\"LIPSYNC_WORK\", \"/content/lipsync_travail\")\nCONFIG = os.environ.get(\"LIPSYNC_CONFIG\", \"configs/unet/stage2.yaml\")\nRUNNER = os.path.join(os.path.dirname(os.path.abspath(__file__)), \"lipsync_runner.py\")\nPROJ_ROOT = os.path.join(BASE, \"projets\")\nos.makedirs(PROJ_ROOT, exist_ok=True)\n\nRES = {\"480p (le plus rapide)\": 480, \"720p (recommandé)\": 720, \"1080p (lent, plus de mémoire)\": 1080}\nQUAL = {\"Rapide (20 étapes)\": 20, \"Meilleure qualité (30 étapes)\": 30}\n\n\ndef slug(name):\n    s = re.sub(r\"[^A-Za-z0-9_-]+\", \"_\", (name or \"\").strip()).strip(\"_\")\n    return s or time.strftime(\"projet_%Y%m%d_%H%M\")\n\n\ndef list_projects():\n    if not os.path.isdir(PROJ_ROOT):\n        return []\n    return sorted(d for d in os.listdir(PROJ_ROOT) if os.path.exists(os.path.join(PROJ_ROOT, d, \"manifest.json\")))\n\n\ndef fmt_time(sec):\n    sec = int(sec)\n    h, m = divmod(sec // 60, 60)\n    return f\"{h} h {m:02d} min\" if h else f\"{m} min {sec % 60:02d} s\"\n\n\ndef _path(f):\n    if f is None:\n        return None\n    return f if isinstance(f, str) else getattr(f, \"name\", None)\n\n\ndef process(name, face, audio, chunk_sec, res_label, qual_label, guidance, resume=False):\n    lines = []\n\n    def log(msg):\n        lines.append(msg)\n\n    def view(head):\n        return f\"### {head}\\n\\n\" + \"\\n\".join(f\"- {l}\" for l in lines[-8:])\n\n    proj = slug(name)\n    proj_dir = os.path.join(PROJ_ROOT, proj)\n    in_dir = os.path.join(proj_dir, \"fichiers_source\")\n    work_dir = os.path.join(WORK, proj)\n\n    if resume:\n        import json\n        mpath = os.path.join(proj_dir, \"manifest.json\")\n        if not os.path.exists(mpath):\n            yield view(\"❌ Projet introuvable.\"), None\n            return\n        m = json.load(open(mpath))\n        face_src, audio_src = m[\"face\"], m[\"audio\"]\n        chunk_sec, max_h = m[\"chunk_sec\"], m[\"max_h\"]\n        if not (os.path.exists(face_src) and os.path.exists(audio_src)):\n            yield view(\"❌ Les fichiers source de ce projet ont disparu. Crée un nouveau projet.\"), None\n            return\n    else:\n        face, audio = _path(face), _path(audio)\n        if not face or not audio:\n            yield view(\"❌ Ajoute un visage (photo ou vidéo) ET un audio.\"), None\n            return\n        os.makedirs(in_dir, exist_ok=True)\n        face_src = os.path.join(in_dir, \"visage\" + os.path.splitext(face)[1].lower())\n        audio_src = os.path.join(in_dir, \"audio\" + os.path.splitext(audio)[1].lower())\n        for old in os.listdir(in_dir):\n            os.remove(os.path.join(in_dir, old))\n        shutil.copy(face, face_src)\n        shutil.copy(audio, audio_src)\n        max_h = RES[res_label]\n\n    steps = QUAL.get(qual_label, 20)\n    log(f\"Projet : **{proj}**\")\n    yield view(\"⏳ Préparation…\"), None\n    try:\n        manifest, jobs = core.prepare_project(proj_dir, work_dir, face_src, audio_src, chunk_sec, max_h, log=log)\n    except Exception as e:\n        log(f\"Erreur pendant la préparation : {e}\")\n        yield view(\"❌ Le fichier n'a pas pu être lu. Vérifie qu'il s'agit bien d'une photo/vidéo et d'un audio.\"), None\n        return\n    yield view(\"⏳ Préparation terminée\"), None\n\n    if jobs:\n        env = dict(os.environ)\n        cmd = [sys.executable, \"-u\", RUNNER, os.path.join(work_dir, \"jobs.json\"),\n               \"--steps\", str(steps), \"--guidance\", str(guidance), \"--config\", CONFIG]\n        p = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                             text=True, bufsize=1)\n        total_chunks = len(manifest[\"chunks\"])\n        already = total_chunks - len(jobs)\n        durations, error, head = [], None, \"⏳ Synchronisation en cours…\"\n        for raw in p.stdout:\n            line = raw.strip()\n            if line.startswith(\"STATUS \"):\n                log(line[7:])\n            elif line.startswith(\"START \"):\n                _, k, n, idx = line.split()\n                done = already + int(k) - 1\n                eta = \"\"\n                if durations:\n                    avg = sum(durations) / len(durations)\n                    eta = f\" · temps restant estimé : **{fmt_time(avg * (int(n) - int(k) + 1))}**\"\n                head = f\"⏳ Morceau {done + 1} / {total_chunks}{eta}\"\n            elif line.startswith(\"DONE \"):\n                durations.append(float(line.split()[3]))\n                log(f\"Morceau terminé en {fmt_time(durations[-1])} (sauvegardé).\")\n            elif line.startswith(\"ERROR FACE\"):\n                error = (\"Aucun visage détecté dans un morceau. Utilise une photo ou une vidéo où le visage \"\n                         \"est bien visible, de face, sans main devant la bouche.\")\n            elif line.startswith(\"ERROR OOM\"):\n                error = (\"Mémoire de la carte graphique pleine. Relance en choisissant une résolution plus \"\n                         \"basse ou des morceaux plus courts.\")\n            elif line.startswith(\"ERROR OTHER\"):\n                error = \"Erreur du modèle : \" + line[12:]\n            elif line.startswith(\"TRACE \"):\n                print(line)  # détail technique visible dans Colab uniquement\n            yield view(head), None\n        p.wait()\n        if error or p.returncode != 0:\n            log(error or \"Le traitement s'est arrêté (voir les détails dans Colab).\")\n            log(\"Les morceaux déjà finis sont gardés : utilise l'onglet « Reprendre » pour continuer.\")\n            yield view(\"❌ Arrêt\"), None\n            return\n\n    final = os.path.join(proj_dir, f\"{proj}_final.mp4\")\n    try:\n        core.assemble(proj_dir, work_dir, manifest, final, log=log)\n    except Exception as e:\n        log(str(e))\n        yield view(\"❌ Assemblage impossible\"), None\n        return\n    shutil.rmtree(work_dir, ignore_errors=True)\n    where = \"dans ton Google Drive, dossier **LipSync/projets/\" + proj + \"**\" if BASE.startswith(\"/content/drive\") \\\n        else \"dans Colab (pense à la télécharger avant de fermer)\"\n    log(f\"Vidéo enregistrée {where}.\")\n    yield view(\"✅ Vidéo prête !\"), final\n\n\ndef new_job(name, face, audio, chunk_sec, res_label, qual_label, guidance):\n    yield from process(name, face, audio, chunk_sec, res_label, qual_label, guidance)\n\n\ndef resume_job(name, qual_label, guidance):\n    if not name:\n        yield \"### ❌ Choisis un projet dans la liste.\", None\n        return\n    yield from process(name, None, None, None, None, qual_label, guidance, resume=True)\n\n\nCSS = \"\"\"\n.gradio-container {max-width: 880px !important; margin: auto}\n#titre h1 {margin-bottom: 0}\n\"\"\"\n\nimport inspect\nTHEME = gr.themes.Soft(primary_hue=\"orange\")\n_STYLE_IN_LAUNCH = \"css\" in inspect.signature(gr.Blocks.launch).parameters\n_blocks_kw = {} if _STYLE_IN_LAUNCH else {\"css\": CSS, \"theme\": THEME}\n_launch_kw = {\"css\": CSS, \"theme\": THEME} if _STYLE_IN_LAUNCH else {}\n\nwith gr.Blocks(title=\"LipSync\", **_blocks_kw) as demo:\n    gr.Markdown(\"# 🎙️ LipSync\\nFais parler une photo ou une vidéo avec ton propre audio — gratuit, \"\n                \"même pour des vidéos de 20 minutes.\", elem_id=\"titre\")\n    with gr.Tab(\"Nouveau projet\"):\n        name = gr.Textbox(label=\"Nom du projet\", placeholder=\"ex : pere_menas_episode_12\")\n        with gr.Row():\n            face = gr.File(label=\"Visage : photo ou vidéo\", file_types=[\"image\", \"video\"])\n            audio = gr.File(label=\"Ton audio (mp3, wav, m4a…)\", file_types=[\"audio\"])\n        with gr.Accordion(\"Réglages (facultatif)\", open=False):\n            chunk = gr.Slider(15, 90, value=30, step=5, label=\"Taille des morceaux (secondes)\",\n                              info=\"30 s est le plus sûr sur Colab gratuit.\")\n            res = gr.Radio(list(RES), value=\"720p (recommandé)\", label=\"Résolution\")\n            qual = gr.Radio(list(QUAL), value=\"Rapide (20 étapes)\", label=\"Qualité\")\n            guid = gr.Slider(1.0, 3.0, value=1.5, step=0.1, label=\"Précision des lèvres\",\n                             info=\"Plus haut = lèvres plus marquées, mais risque de tremblement.\")\n        go = gr.Button(\"🎬 Lancer la synchronisation\", variant=\"primary\", size=\"lg\")\n        status = gr.Markdown()\n        result = gr.File(label=\"Vidéo finale (clique pour télécharger)\")\n        go.click(new_job, [name, face, audio, chunk, res, qual, guid], [status, result])\n\n    with gr.Tab(\"Reprendre un projet\"):\n        gr.Markdown(\"Si Colab s'est coupé, relance l'outil puis choisis ton projet : \"\n                    \"les morceaux déjà faits ne sont pas refaits.\")\n        with gr.Row():\n            proj = gr.Dropdown(list_projects(), label=\"Projet\", scale=4)\n            refresh = gr.Button(\"🔄\", scale=1)\n        qual2 = gr.Radio(list(QUAL), value=\"Rapide (20 étapes)\", label=\"Qualité\")\n        guid2 = gr.Slider(1.0, 3.0, value=1.5, step=0.1, label=\"Précision des lèvres\")\n        go2 = gr.Button(\"▶️ Reprendre\", variant=\"primary\", size=\"lg\")\n        status2 = gr.Markdown()\n        result2 = gr.File(label=\"Vidéo finale\")\n        refresh.click(lambda: gr.Dropdown(choices=list_projects()), None, proj)\n        go2.click(resume_job, [proj, qual2, guid2], [status2, result2])\n\n    gr.Markdown(\"<small>Utilise des visages générés par IA, le tien, ou ceux de personnes qui ont donné leur accord.</small>\")\n\nif __name__ == \"__main__\":\n    demo.queue().launch(share=os.environ.get(\"LIPSYNC_SHARE\", \"1\") == \"1\", debug=True,\n                        allowed_paths=[BASE], server_port=int(os.environ.get(\"PORT\", \"7860\")), **_launch_kw)\n"}
for name, code in SOURCES.items():
    open(os.path.join(APP, name), "w").write(code)

# 6. Lancement
os.environ.update({"LIPSYNC_REPO": REPO, "LIPSYNC_BASE": BASE, "LIPSYNC_CONFIG": CONFIG,
                   "LIPSYNC_WORK": "/content/lipsync_travail"})
sys.path.insert(0, APP)
for m in ("lipsync_app", "lipsync_core"):
    sys.modules.pop(m, None)
import lipsync_app
etape("🚀 Clique sur le lien « Running on public URL » ci-dessous (https://….gradio.live)")
lipsync_app.demo.queue().launch(share=True, debug=True, allowed_paths=[BASE], **lipsync_app._launch_kw)
